# LNR.TO — Your Starter Agent

**If you're not sure what to do next, continue from here.**

This notebook is a fresh, hackable agent for Linamar Corporation (`LNR.TO`), following the S&P 500 starter-agent flow:

- **optional news search** — bounded, cutoff-aware Google Search (proxy-only)
- **optional code execution** — an E2B Python sandbox
- **one lightweight forecasting skill** — a tool-usage playbook in `starter_agent/skills/`

It lets you talk to the agent (Track 2) and score one real forecast (Track 1). Live cells are gated by `RUN_AGENT` so a fresh Run All is safe and free; flip it to `True` to call the model.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "aieng-forecasting").is_dir()), Path.cwd().resolve())
sys.path[:0] = [str(ROOT / "aieng-forecasting"), str(ROOT / "implementations")]
load_dotenv(ROOT / ".env", override=False)

AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"  # advanced (higher cost/latency)
RUN_AGENT = False

from LNR_Forecasting.starter_agent import (
    build_starter_agent_config,
    build_starter_agent_predictor,
)

print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL)

RUN_AGENT = True | model = gemini-3.1-flash-lite-preview


---
## 1. Meet your agent

`build_starter_agent_config` returns an `AgentConfig` with two toggles. The default turns **news search on** (proxy-only, no extra key) and **code execution off** (it needs `E2B_API_KEY` and is slower). Flip them and re-run — the loaded skills follow the enabled tools.

In [7]:
config = build_starter_agent_config(
    model=AGENT_MODEL,
    enable_search=True,
    enable_code_exec=False,
)

print("Agent:", config.name)
print("Search enabled:    ", config.context_retrieval.enabled)
print("Code-exec enabled: ", config.code_execution.enabled)
print("Skills loaded:     ", [p.name for p in config.skills_dirs])
print("\n── System instruction (edit this in starter_agent/agent.py) ──\n")
print(config.instruction[:1200], "...")

Agent: lnr_starter_agent
Search enabled:     True
Code-exec enabled:  False
Skills loaded:      ['forecasting', 'research-playbook']

── System instruction (edit this in starter_agent/agent.py) ──

You are a transparent equity-market analyst focused on Linamar Corporation (LNR.TO), a Canadian industrial and auto-supply company. Consider rates, Canadian and US auto production, tariffs, industrial demand, CAD/USD, commodities, and broad risk sentiment. For conversation, answer directly and concisely. When asked for a structured forecast, produce a calibrated probabilistic forecast from the supplied evidence and do not invent facts. ...


---
## 2. Talk to it  *(Track 2 — open-ended analysis)*

Ask the agent anything. This is the interactive mode: no scoring, no schema — just reasoning (and a web search, since search is on). Edit the question and explore.

In [8]:
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig


QUESTION = (
    "What are the key macro, industrial, auto-sector, tariff, currency, and commodity risks "
    "for Linamar (LNR.TO) over the next month, and how should they shape the spread of a "
    "1-week LNR.TO return forecast? Be concise."
)

if RUN_AGENT:
    chat_agent = build_adk_agent(config)
    runner = AdkTextRunner(chat_agent, config=AdkTextRunnerConfig(app_name="lnr_starter_chat"))
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, PLE1142
    print(reply)
else:
    print("RUN_AGENT is False — set it to True in the setup cell to talk to the agent.")

For the next month, LNR.TO is sensitive to the interplay between cyclical industrial demand and auto-sector volatility.

### **Key Risks**
*   **Macro/Auto:** Continued sensitivity to U.S. auto production volumes and interest-rate-driven financing constraints on the Industrial segment (Skyjack/Agriculture).
*   **Tariffs:** Lingering uncertainty regarding USMCA trade status and potential Section 232 metal-derivative levies, though Linamar remains largely USMCA-compliant.
*   **Currency:** A strengthening USD against the CAD typically acts as a net tailwind for Linamar’s export-heavy revenue, but volatility creates margin-hedging noise.
*   **Commodities:** Fluctuations in steel/aluminum prices directly impact COGS (Cost of Goods Sold), while low agricultural commodity prices suppress capital investment in that segment.
*   **Event Risk:** The November 4, 2026, Q3 earnings release is the primary catalyst for immediate price action.

### **1-Week Return Forecast Spread**
The distribution

Root node lnr_starter_agent was cancelled.


---
## 3. Score one prediction against a known outcome  *(Track 1)*

Now run the agent as a `Predictor`. We pick the most recent origin whose horizon has already resolved, forecast the 1-week LNR.TO return, and check whether the actual return landed inside the agent's 80% band. One origin cannot tell you if the agent is calibrated; use a broader backtest for that. Live, so gated by `RUN_AGENT`.

In [9]:
from datetime import datetime, timezone


if RUN_AGENT:
    from aieng.forecasting.evaluation.task import ForecastingTask
    from LNR_Forecasting import build_lnr_multivariate_service
    from LNR_Forecasting.data import lnr_logret_series_id

    HORIZON = 5
    COVARIATES = ["vix_level_l1b", "oil_log_ret_1b_l1b", "nasdaq_log_ret_1b_l1b"]
    svc = build_lnr_multivariate_service(covariate_series_ids=COVARIATES)
    now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
    tgt = lnr_logret_series_id(HORIZON)
    full = svc.get_series(tgt, as_of=now)
    full["timestamp"] = pd.to_datetime(full["timestamp"])
    last_date = full["timestamp"].iloc[-1]
    AS_OF = last_date - pd.offsets.BDay(HORIZON + 1)

    task = ForecastingTask(
        task_id=f"lnr_logret_{HORIZON}b",
        target_series_id=tgt,
        horizons=[HORIZON],
        frequency="B",
        description=f"LNR.TO cumulative log return, {HORIZON} business days ahead (starter).",
    )
    ctx = svc.context(as_of=AS_OF)
    pred = build_starter_agent_predictor(config, covariate_series_ids=COVARIATES).predict(task, ctx)[0]

    rows = full[full["timestamp"] >= AS_OF + pd.offsets.BDay(HORIZON)]
    actual = float(rows["value"].iloc[0]) if not rows.empty else None
    fc = pred.payload
    lo, hi = fc.quantiles[0.10], fc.quantiles[0.90]
    print(f"Origin as_of={AS_OF.date()}  horizon={HORIZON}b (1 week)  (latest data {last_date.date()})\n")
    print(f"  agent point  : {fc.point_forecast:+.4f}  ({fc.point_forecast * 100:+.2f}%)")
    print(f"  agent 80% CI : [{lo:+.4f}, {hi:+.4f}]")
    if actual is None:
        print("  actual       : N/A (not yet resolved)")
    else:
        in_band = "yes" if lo <= actual <= hi else "no"
        print(f"  actual       : {actual:+.4f}  ({actual * 100:+.2f}%)   in 80% band? {in_band}")
    if pred.metadata.get("rationale"):
        print("\nRationale:", pred.metadata["rationale"][:300])
else:
    print("RUN_AGENT is False — set it to True to score a live forecast against a known outcome.")

Origin as_of=2026-09-14  horizon=5b (1 week)  (latest data 2026-09-22)

  agent point  : +0.0015  (+0.15%)
  agent 80% CI : [-0.0480, +0.0550]
  actual       : -0.0180  (-1.80%)   in 80% band? yes

Rationale: Linamar (LNR.TO) has exhibited significant volatility in its recent log returns. Following a period of sharp decline in late August, the stock showed signs of stabilization in early September. The forecast model incorporates these trends, using a distribution centered near zero to account for potent


---
## 4. Make it yours

This agent is a starting point. Here are concrete next steps, easiest first — each is a small edit, then re-run the cells above.

1. **Flip code execution on.** Set `enable_code_exec=True` in §1 (needs `E2B_API_KEY`). The agent loads the `code-analysis-playbook` skill and can compute its own LNR diagnostics before forecasting.
2. **Edit the agent's personality.** Open `starter_agent/agent.py` and change `_starter_instruction()` — make it more cautious, more contrarian, or focused on industrial demand. Re-run §1 to see the new instruction.
3. **Sharpen the skills.** The files in `starter_agent/skills/` are short on purpose. Add research queries or return diagnostics; the agent picks them up automatically.
4. **Change the question and the origin.** Try a different `QUESTION` in §2 and a different origin in §3.
5. **Widen the covariate panel.** Pass the full `DEFAULT_COVARIATE_SERIES_IDS` list to both the service and `build_starter_agent_predictor(...)` and compare the context.
6. **Forecast other horizons.** Swap `HORIZON` to 1 (next session) or 21 (one month) — each maps to its own `lnr_logret_{h}b` target.

For the conventional numerical comparison, use the shared `DataService` from `LNR_Forecasting.data` with predictors from `aieng.forecasting.methods.numerical`.